## 3、长期记忆

对应《尚硅谷-09-上下文与记忆.pdf》第 3 章，PDF 第 38-60 页（到第 4 章标题之前）。

沿用上一份笔记的格式，按照课件顺序整理：**基本理解 → Store 的读写与检索 → 工具和中间件中访问 → 何时写入记忆**。
保留课件中的 `store`、`namespace`、`key`、`value`、`CustomState`、`save_user_info` 和 `get_user_info` 等命名。

**运行方法**：选择项目的 `langchain` Python 内核。先运行“公共导入”，再按小节练习。

| 示例 | 是否访问外部服务 | 前置准备 |
| --- | --- | --- |
| 内存 Store 的读写、前缀搜索、过滤、固定向量 | 不访问 | 现有 `langchain` 环境 |
| 真实 Embedding 语义检索 | 调用嵌入模型 | 该小节指定的模型服务配置 |
| 工具和中间件中的 Agent | 调用聊天模型 | 项目 `.env` 中的百炼配置 |
| PostgreSQL Store | 连接数据库；包含 Agent 时还会调用模型 | 自己的 `DB_URL` |

各小节会重新创建 `store` 和 `agent`，只运行调用语句可复用已有对象，重跑初始化会创建新对象。
文中的预期现象是观察指引，不是本次真实模型或数据库的执行结果。


### 3.1 基本理解

#### 3.1.1 什么是长期记忆

短期记忆保存某个会话的 State；长期记忆通过 Store 组织用户、组织或应用的数据，让不同会话可以按相同位置读写。
例如：用户喜欢简洁回答、常用 Python、个人资料，以及过去任务中的有效经验。

| 对比 | 短期记忆 | 长期记忆 |
| --- | --- | --- |
| 主要保存什么 | 当前会话历史和运行状态 | 用户偏好、事实、经验、规则 |
| 组织方式 | Checkpointer + `thread_id` | Store + `namespace` + `key` |
| 不同会话是否自动共享 | 不同线程的状态分开 | 应用可以让不同会话访问同一条 Store 数据 |
| 内存实现 | `InMemorySaver` | `InMemoryStore` |
| PostgreSQL 实现 | `PostgresSaver` | `PostgresStore` |

**“长期”指数据的作用域和用途，不保证一定落盘。** `InMemoryStore` 也用于长期记忆的开发演示，但重建实例或重启内核仍会丢失其内存数据。
模型不会自动把整段聊天转成长期记忆；应用需要决定保存什么，并在合适时机读取、提供给模型。

#### 3.1.2 类型划分

| 类型 | 记录什么 | 课件举例 |
| --- | --- | --- |
| Semantic Memory（语义记忆） | 事实、概念、偏好 | 用户喜欢简洁回答、常用中文 |
| Episodic Memory（情景记忆） | 过去的任务经验 | 成功任务的输入与输出，可用作 few-shot examples |
| Procedural Memory（程序性记忆） | 规则、做事方法 | 系统提示词、工作流程、工具调用规则 |

这里的“语义记忆”是一类记忆内容；后面的“语义搜索”是一种检索方法，二者不是同一个概念。


#### 3.1.3 存储架构

**Store → Namespace → Key → Value**。

| 层级 | 类型 / 示例 | 含义 |
| --- | --- | --- |
| Store | `InMemoryStore()`、`PostgresStore` | 保存和检索数据的仓库，实现 `BaseStore` 接口 |
| Namespace | `("users", "user_123", "preferences")` | 字符串元组组成的层级路径 |
| Key | `"profile"` | 该命名空间内一条数据的唯一键 |
| Value | `{"language": "zh-CN", "style": "short_direct"}` | 保存的 JSON-like 字典 |

同一个 Store 可以被多个会话共用，再用 namespace 或 key 区分用户。
`namespace + key` 一起定位一条数据：不同 namespace 可以出现相同的 key。
只有一个元素的元组要写为 `("users",)`，末尾逗号不能省略。

```text
会话 t1 的 State ─┐
会话 t2 的 State ─┼─ 应用按用户标识读写同一个 Store
会话 t3 的 State ─┘
                  ├─ ("users", "user_1", "memories")
                  │    ├─ "profile" → {"name": "张三", "city": "上海"}
                  │    └─ "writing_style" → {"language": "zh-CN"}
                  └─ ("users", "user_2", "memories")
                       └─ "profile" → {"name": "李四", "city": "深圳"}
```

不同会话要读取同一份资料，需要使用相同的保存后端、namespace 和 key；“跨会话”不表示所有用户的数据都应该混在一起。


### 公共导入

基础 Store 示例可以离线运行，不需要先初始化聊天模型。
PostgreSQL 和模型示例会在各自的小节读取 `.env`。


In [3]:
import os
from pprint import pprint

from dotenv import load_dotenv
from rich import print as rprint
from langgraph.store.memory import InMemoryStore


### 3.2 基础 API 的使用

可以在 Agent 运行图之外直接读写 Store，不必每次都调用大模型。

| 方法 | 作用 | 返回值 |
| --- | --- | --- |
| `store.put(namespace, key, value)` | 写入；同 namespace、同 key 时更新该条数据 | `None` |
| `store.get(namespace, key)` | 精确读取一条数据 | `Item`；不存在时为 `None` |
| `store.search(namespace_prefix, ...)` | 在命名空间前缀下检索 | `list[SearchItem]` |

#### 3.2.1 put()/get()：写入 / 读取 API

`put()` 的主要参数：

| 参数 | 含义 |
| --- | --- |
| `namespace` | 字符串元组表示的层级路径 |
| `key` | 该路径内的数据键 |
| `value` | 要保存的字典 |
| `index=None` | 使用 Store 初始化时的语义索引配置；没有索引配置时不生成向量 |
| `index=False` | 该条数据不生成语义索引，但仍会保存 value |
| `index=["course"]` | 对本条数据指定的字段建立语义索引，需要 Store 已配置嵌入能力 |
| `ttl` | 可选的存活时间，支持情况取决于后端；当前 `InMemoryStore` 不支持 TTL |

`get()` 返回的 `Item` 不只有 value，还有 `namespace`、`key`、`created_at`、`updated_at`。
`get(..., refresh_ttl=...)` 可控制是否刷新 TTL；本章不配置 TTL。

**① 基于 InMemoryStore：添加并读取**


In [4]:
store = InMemoryStore()

namespace = ("users",)
user_id = "user-1"
username = "小蓝"

store.put(namespace, user_id, {"name": username})
item = store.get(namespace, user_id)

rprint(item)
if item is not None:
    print("value =", item.value)
    print("name =", item.value["name"])
    print("created_at =", item.created_at)
    print("updated_at =", item.updated_at)

print("不存在的 key：", store.get(namespace, "not-exist"))


Item(namespace=['users'], key='user-1', value={'name': '小蓝'}, created_at='2026-10-21T06:48:44.057570+00:00', 
updated_at='2026-10-21T06:48:44.057576+00:00')

value = {'name': '小蓝'}
name = 小蓝
created_at = 2026-10-21 06:48:44.057570+00:00
updated_at = 2026-10-21 06:48:44.057576+00:00
不存在的 key： None


**② 更新：namespace 和 key 都不变，只改变 value。**

再次 `put()` 会把这条记录的 value 替换为新的字典，不会自动进行字段级合并。
下面把“小蓝”更新为“小红”，再比较前后 Item。


In [5]:
old_item = store.get(namespace, user_id)

store.put(namespace, user_id, {"name": "小红"})
new_item = store.get(namespace, user_id)

print("更新前：")
rprint(old_item)
print("更新后：")
rprint(new_item)


更新前：


Item(namespace=['users'], key='user-1', value={'name': '小蓝'}, created_at='2026-10-21T06:48:44.057570+00:00', 
updated_at='2026-10-21T06:48:44.057576+00:00')

更新后：


Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-10-21T06:48:46.696864+00:00', 
updated_at='2026-10-21T06:48:46.696868+00:00')

**当前版本的时间字段**：`InMemoryStore` 每次 `put()` 都会创建新的 Item，因此同 key 更新时 `created_at` 也会刷新。
课件说 `created_at` 和 `updated_at`“始终一致”，更准确地说是二者都在本次写入时生成，数值可能相差几个微秒，不保证严格相等。

**③ 基于 PostgresStore：添加和读取（需要数据库，按需运行）**

使用项目 `.env` 中自己的 `DB_URL`。值只填写 `postgresql://...` 连接地址，不能把终端的 `psql "..."` 命令一起填进去。
项目已有 PostgreSQL Store 所需依赖，不重复安装。


In [ ]:
from langgraph.store.postgres import PostgresStore

load_dotenv(override = True)
DB_URL = os.environ["DB_URL"]

namespace = ("users",)
user_id = "user-11"
username = "小蓝"

with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    store.put(namespace, user_id, {"name": username})
    print(store.get(namespace, user_id))


**④ 重新连接数据库，更新同一条数据。**

这里仍用同一个 `DB_URL`、namespace 和 key。
课件更新示例中混用了 `DB_URL`、`DB_URI`，本笔记统一为 `DB_URL`。


In [ ]:
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    old_item = store.get(namespace, user_id)
    store.put(namespace, user_id, {"name": "小红"})
    new_item = store.get(namespace, user_id)

    print("更新前：")
    rprint(old_item)
    print("更新后：")
    rprint(new_item)


| 同 namespace、同 key 再次 put | 当前 InMemoryStore | PostgresStore |
| --- | --- | --- |
| value | 替换为新字典 | 更新为新字典 |
| created_at | 随本次写入重新生成 | 保留原创建时间 |
| updated_at | 随本次写入重新生成 | 更新为本次写入时间 |
| 重建实例或重启内核 | 原实例内存数据不再可用 | 重新连接同一数据库可读取已保存数据 |

`PostgresStore.setup()` 用于初始化 Store 的表结构和迁移；不会因为重新调用就清空已有资料。
退出 `with` 后该连接会关闭，依赖连接的 Store 操作放在 `with` 块内。


#### 3.2.3 search()：检索 API

沿用课件编号；课件在 `3.2.1` 后直接标为 `3.2.3`。

| 参数 | 含义 |
| --- | --- |
| `namespace_prefix` | 按层级前缀选择搜索范围，作为第一个位置参数传入 |
| `query` | 用于向量相似度检索的查询文本，需要配置语义索引 |
| `filter` | 按 value 的字段值做结构化过滤 |
| `limit` | 最多返回多少条，默认 10 |
| `offset` | 跳过前多少条，默认 0 |
| `refresh_ttl` | 是否刷新 TTL，本章不配置 |

搜索结果是 `SearchItem` 列表；除 Item 的基本字段外，还有 `score`。
未进行语义检索时，下面示例的 `score` 为 `None`。

**① 准备课件中的三份用户偏好。**


In [8]:
store = InMemoryStore()

namespace1 = ("users", "Alice", "memories")
key1 = "preferences"
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶",
}

namespace2 = ("users", "Bob", "memories")
key2 = "preferences"
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦",
}

namespace3 = ("users", "Black", "memories")
key3 = "preferences"
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶",
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)


**② 按 namespace 前缀搜索。**

`("users",)` 覆盖三个人；`("users", "Alice")` 只覆盖 Alice 路径下的数据。
这是元组层级的前缀匹配，不是对用户名称做模糊搜索。


In [ ]:
print("=" * 30, "-> (users,) <-", "=" * 30)
for item in store.search(("users",)):
    print(item)

print("=" * 30, "-> (users, Alice) <-", "=" * 30)
for item in store.search(("users", "Alice")):
    print(item)


**③ 按 filter 过滤 value。**

在当前三份示例数据中，预期匹配关系如下（不依赖结果顺序）：

| 条件 | 匹配用户 |
| --- | --- |
| `{"food": "紫光园奶皮子酸奶"}` | Alice、Black |
| `{"sports": "跑步"}` | Alice、Bob |
| `{"course": "数字电路与模拟电路"}` | Bob、Black |

这里比较的是实际字段值，因此示例字符串统一去掉了 PDF 排版带来的首尾空格。


In [6]:
print("=" * 30, "-> filter=food <-", "=" * 30)
for item in store.search(("users",), filter = {"food": "紫光园奶皮子酸奶"}):
    print(item)

print("=" * 30, "-> filter=sports <-", "=" * 30)
for item in store.search(("users",), filter = {"sports": "跑步"}):
    print(item)

print("=" * 30, "-> filter=course <-", "=" * 30)
for item in store.search(("users",), filter = {"course": "数字电路与模拟电路"}):
    print(item)


============================== -> filter=food <- ==============================
============================== -> filter=sports <- ==============================
============================== -> filter=course <- ==============================


**④ 按语义搜索：先用自定义嵌入函数观察向量。**

本例继续使用上面的 `namespace1`、`key1`、`value1` 等三组数据。
课件的 `embed()` 无论输入什么文本，都返回 6 维全 1 向量，目的是观察索引结构，不具备真正区分文本含义的能力。

| index 配置 | 含义 |
| --- | --- |
| `embed` | 把一批文本转换成对应的一批向量的函数，或 Embeddings 对象 |
| `dims` | 向量维度，应与嵌入结果一致 |
| `fields` | 从 value 中提取哪些内容生成向量 |

`fields=["$"]` 表示整个 value；`["course"]` 表示课程字段；本例同时用 `["$", "course"]` 观察两份字段向量。
还可使用嵌套路径，如 `"parent.child"` 或 `"array[*].field"`；数组路径可能提取多段文本。


In [9]:
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]


index_config = {
    "embed": embed,
    "dims": 6,
    "fields": ["$", "course"],
}

store = InMemoryStore(index = index_config)
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

pprint(store._vectors)
print("Alice 的整个 value 对应的向量：")
pprint(store._vectors[namespace1][key1]["$"])


defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x000001940B138C20>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {'preferences': {'$': [1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0],
                                                                          'course': [1.0,
                                                                                     1.0,
                                                                                     1.0,
                                                           

In [10]:
for item in store.search(("users",), query = "数电模电"):
    print(item)


Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-21T06:51:14.678130+00:00', updated_at='2026-10-21T06:51:14.678134+00:00', score=1.0000000000000002)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-21T06:51:14.678325+00:00', updated_at='2026-10-21T06:51:14.678327+00:00', score=1.0000000000000002)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-21T06:51:14.678448+00:00', updated_at='2026-10-21T06:51:14.678450+00:00', score=1.0000000000000002)


因为查询文本和所有候选文本都被转换成相同向量，结果分数会相同或仅有浮点表示差异，不能据此判断谁更相关。
`_vectors` 是当前内存实现的内部属性，这里仅沿用课件观察；业务读写仍使用公开的 `put/get/search`。

**⑤ 使用真实嵌入模型（会请求 Embedding 服务，按需运行）。**

本例沿用课件的 `openai:text-embedding-3-large`、`CLOSEAI_API_KEY` 和 `CLOSEAI_BASE_URL`，向量维度配置为 3072。
当前项目 `.env` 尚未配置这两个 `CLOSEAI_*` 变量；先补齐可调用该模型的服务配置，再运行本单元。
这与后面聊天模型所用的百炼配置是两组不同用途的配置，不自动互换。

创建模型客户端后，下面的 `store.put(...)` 就会调用嵌入服务生成文档向量，`store.search(..., query=...)` 还会生成查询向量。


In [11]:
from langchain.embeddings import init_embeddings

load_dotenv(override = True)
embedding_model = init_embeddings(
    model = "openai:text-embedding-3-large",
    api_key = os.environ["CLOSEAI_API_KEY"],
    base_url = os.environ["CLOSEAI_BASE_URL"],
)

index_config = {
    "embed": embedding_model,
    "dims": 3072,
    "fields": ["$"],
}

store = InMemoryStore(index = index_config)
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

for item in store.search(("users",), query = "数电模电"):
    print(item)


KeyError: 'CLOSEAI_API_KEY'

课件的真实模型输出中，Bob、Black 的“数字电路与模拟电路”比 Alice 的课程更接近查询“数电模电”。
实际排名和分数以所用嵌入模型的返回为准，不把课件数值写成固定答案。

**query 主要用于排序，不是“只返回包含这些字的记录”。** 本例默认 `limit=10`，只有 3 条候选，所以可能全部返回。
要控制条数可使用 `limit`，限定人群或课程等结构化条件可结合 `filter`；它们与语义相似度各自承担不同作用。


### 3.3 在 Agent 运行图中访问长期记忆

前面直接用 Python 调用 Store；这一节把 Store 绑定到 Agent，让工具和中间件在运行期间访问它。
`create_agent(..., store=store)` 负责提供 Store，具体的读取、写入仍由相应代码执行。

**聊天模型准备**：与上一章和现有工程一致，使用百炼环境变量初始化 `model`。
本单元只创建客户端，后面的 `agent.invoke(...)` 才会发送聊天请求。


In [13]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage
from langchain.tools import tool, ToolRuntime
from typing import NotRequired

load_dotenv(override = True)
model = init_chat_model(
    model = "openai:qwen3.8-max",
    api_key = os.getenv("BAILIAN_API_KEY"),
    base_url = os.getenv("BAILIAN_API_URL"),
)


#### 3.3.1 在工具中访问长期记忆

**① 基于 InMemoryStore：定义状态与工具。**

`CustomState` 在默认 `messages` 之外增加 `user_id`。
框架向工具注入 `ToolRuntime`，工具用 `runtime.store` 访问 Store，用 `runtime.state["user_id"]` 找到本次用户。

这里沿用课件的另一种组织方式：namespace 固定为 `("users",)`，把 `user_id` 用作 key。
模型只需为 `save_user_info` 提供姓名；`runtime` 不会成为需要模型填写的工具参数。

`NotRequired[str]` 表示该键在类型结构上可缺省，但本例工具直接读取它，因此下面每次调用都必须传入 `user_id`。


In [14]:
class CustomState(AgentState):
    user_id: NotRequired[str]


@tool(parse_docstring = True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """将用户信息保存在长期记忆中。

    Args:
        name: 用户名。

    Returns:
        保存状态。
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"


@tool(parse_docstring = True)
def get_user_info(runtime: ToolRuntime) -> str:
    """从长期记忆中读取用户信息。

    Returns:
        已保存的用户信息，不存在时返回 unknown。
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item is not None else "unknown"


In [15]:
store = InMemoryStore()

agent = create_agent(
    model = model,
    tools = [save_user_info, get_user_info],
    store = store,
    system_prompt = "用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索。",
    state_schema = CustomState,
)

print("=" * 30, "-> 第一次独立调用 <-", "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1",
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, "-> 第二次独立调用 <-", "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1",
})
for msg in response2["messages"]:
    msg.pretty_print()

print("\n直接检查 Store 中是否写入成功：")
rprint(store.get(("users",), "user-1"))


============================== -> 第一次独立调用 <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_6c08859d31254d35ab82f57b)
 Call ID: call_6c08859d31254d35ab82f57b
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小花！很高兴认识你 😊

我已经记住你的名字了。有什么我可以帮你的吗？无论是想聊聊天，还是有具体的问题需要解决，都可以告诉我。
============================== -> 第二次独立调用 <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================

我来帮您查一下。
Tool Calls:
  get_user_info (call_0b84c0b9bf184ec9a7f79827)
 Call ID: call_0b84c0b9bf184ec9a7f79827
  Args:
=====

Item(namespace=['users'], key='user-1', value={'name': '小花'}, created_at='2026-10-21T07:02:46.811347+00:00', 
updated_at='2026-10-21T07:02:46.811351+00:00')

预期流程：

```text
第一次调用：我是小花
    → 模型调用 save_user_info(name="小花")
    → store.put(("users",), "user-1", {"name": "小花"})

第二次调用：我是谁
    → 模型调用 get_user_info()
    → store.get(("users",), "user-1")
    → 工具结果进入本次消息 → 模型据此回答
```

本例没有配置 checkpointer，也没有传 `thread_id`；两次调用各自从新消息开始。
第二次能得到姓名，是因为它重新读取了同一个 Store 中的资料，而不是自动继承第一轮聊天记录。
课件称其为“两个会话（线程）”，这里更严格地说是两次未持久化的独立调用。

**查看证据**：是否出现 `save_user_info/get_user_info` 的工具调用与结果，以及 `store.get(...)` 是否确实返回了资料。
模型可能不按预期选择工具，不能只凭一句“小花”就判定存取成功。
重新创建一个空 `InMemoryStore` 后，旧姓名不会自动恢复；换用户 key 则读取另一份资料。


**② 基于 PostgresStore（需要数据库和聊天模型，按需运行）。**

复用上面定义的 `CustomState` 和两个工具，只把保存后端换成 PostgreSQL。
Store 通过运行时注入，因此工具内部不需要固定绑定某个内存对象。
数据库中同一个 `("users",) + "user-1"` 可能已有资料；再次运行会读取或更新该位置。


In [16]:
from langgraph.store.postgres import PostgresStore

load_dotenv(override = True)
DB_URL = os.environ["DB_URL"]

with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    agent = create_agent(
        model = model,
        tools = [save_user_info, get_user_info],
        store = store,
        system_prompt = "用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索。",
        state_schema = CustomState,
    )

    print("=" * 30, "-> 第一次独立调用 <-", "=" * 30)
    response1 = agent.invoke({
        "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
        "user_id": "user-1",
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, "-> 第二次独立调用 <-", "=" * 30)
    response2 = agent.invoke({
        "messages": [HumanMessage("我是谁")],
        "user_id": "user-1",
    })
    for msg in response2["messages"]:
        msg.pretty_print()

    print("\n数据库 Store 中的资料：")
    rprint(store.get(("users",), "user-1"))


============================== -> 第一次独立调用 <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你 😊

我先把你的名字记下来，方便以后记住你。
Tool Calls:
  save_user_info (call_094af33586394e518a52bd9d)
 Call ID: call_094af33586394e518a52bd9d
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小花！很高兴认识你 😊 我已经记住你的名字啦。

有什么我可以帮你的吗？无论是回答问题、写东西、翻译，还是单纯聊聊天，都可以随时告诉我～
============================== -> 第二次独立调用 <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_ee0eda8f76304d96904e4871)
 Call ID: call_ee0eda

Item(namespace=['users'], key='user-1', value={'name': '小花'}, created_at='2026-10-21T15:03:00.479464+08:00', 
updated_at='2026-10-21T15:03:00.479464+08:00')

在 `psql` 客户端中可使用只读命令查看表（不在 Python 单元中执行）：

```sql
\dt
SELECT prefix, key, value, created_at, updated_at
FROM store
WHERE key = 'user-1';
```

本章未配置 PostgreSQL 向量索引时，Store 主要使用 `store` 和 `store_migrations` 表。
上一章 `PostgresSaver` 使用的 `checkpoints`、`checkpoint_blobs`、`checkpoint_writes` 等是检查点表。
即使二者连接同一数据库，也分别负责长期资料与会话状态，不会自动相互转换。


#### 3.3.2 在中间件中访问长期记忆

| 位置 | 读取 Store 的入口 | 读取本例用户 ID |
| --- | --- | --- |
| Node-style，如 `before_model` | `runtime.store` | `state["user_id"]` |
| Wrap-style：`wrap_model_call` | `request.runtime.store` | `request.state["user_id"]` |
| Wrap-style：`wrap_tool_call` | `request.runtime.store` | `request.state["user_id"]` |

`Runtime` 和 `ToolRuntime` 名称相近，但分别用于图运行上下文和工具运行上下文。
Store 都来自创建 Agent 时传入的 `store`。

课件本小节主要列接口定义。下面用一个最小观察示例串起三个入口：中间件读取并打印资料，两个工具仍负责实际存取。
仅把资料打印到日志不会自动把它提供给模型；本例由 `get_user_info` 的工具结果把资料带入消息。


In [17]:
from typing import Any, Callable
from langchain.agents.middleware import (
    before_model,
    wrap_model_call,
    wrap_tool_call,
    ModelRequest,
    ModelResponse,
)
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import ToolMessage
from langgraph.runtime import Runtime
from langgraph.types import Command


@before_model
def read_store_before_model(state: CustomState, runtime: Runtime) -> None:
    item = runtime.store.get(("users",), state["user_id"])
    print("[before_model]", item.value if item is not None else "unknown")
    return None


In [18]:
@wrap_model_call
def read_store_in_model_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    item = request.runtime.store.get(("users",), request.state["user_id"])
    print("[wrap_model_call]", item.value if item is not None else "unknown")
    return handler(request)


In [19]:
@wrap_tool_call
def read_store_in_tool_call(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    item = request.runtime.store.get(("users",), request.state["user_id"])
    print("[wrap_tool_call]", item.value if item is not None else "unknown")
    return handler(request)


In [20]:
store = InMemoryStore()
store.put(("users",), "user-1", {"name": "小花"})

agent = create_agent(
    model = model,
    tools = [save_user_info, get_user_info],
    store = store,
    state_schema = CustomState,
    system_prompt = "用户询问个人信息时，调用 get_user_info 读取资料再回答。",
    middleware = [
        read_store_before_model,
        read_store_in_model_call,
        read_store_in_tool_call,
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1",
})

for msg in response["messages"]:
    msg.pretty_print()


[before_model] {'name': '小花'}
[wrap_model_call] {'name': '小花'}
[wrap_tool_call] {'name': '小花'}
[before_model] {'name': '小花'}
[wrap_model_call] {'name': '小花'}
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_130d2a293154417c991ee093)
 Call ID: call_130d2a293154417c991ee093
  Args:
================================= Tool Message =================================
Name: get_user_info

{'name': '小花'}
================================== Ai Message ==================================

根据我记住的信息，你是**小花** 🌸

如果这个名字需要更新，告诉我新的称呼就行，我会帮你改过来。


预期先在 `before_model`、`wrap_model_call` 中观察 Store，模型决定调用工具后进入 `wrap_tool_call`。
工具返回后还会再次调用模型，因此模型相关 hook 可能执行多次。
Wrap-style 中的 `handler(request)` 用来继续执行被包裹的模型或工具调用。

如果只需要读取偏好并调整本次模型提示，也可以在合适的模型请求中使用查到的资料；本例聚焦课件的 Store 访问入口，不额外修改提示词或消息列表。


### 3.4 何时写入记忆

| 方式 | 在什么时候写 | 优点 | 代价 | 课件中的适用场景 |
| --- | --- | --- | --- | --- |
| Hot path（主流程） | 处理本次请求时同步判断并保存 | 写入成功后，后续请求可立即读取 | 增加本次流程的工作和延迟 | 用户偏好、账号资料 |
| Background（后台） | 先回答，再由后台任务整理保存 | 主流程更快，便于批量整理 | 需要安排触发时机；未完成前读取不到新资料 | 对话摘要、经验沉淀、行为分析 |

上面的 `save_user_info` 工具属于主流程写入：工具执行时调用 `store.put()`，写入成功后才返回 `"saved"`。
后台写入需要应用另外安排后台任务；只传 `store=store` 不会自动开启后台整理。
本章只比较这两种时机，不额外实现后台调度。

### 本章回顾

- Store 提供数据仓库；namespace 与 key 决定“哪一份资料”，value 是具体内容。
- `put/get` 用于写入与精确读取，`search` 用于前缀范围、结构化过滤和配置索引后的语义检索。
- `InMemoryStore` 可以演示跨调用共享，但要跨内核重启保留资料，需要持久保存后端。
- 工具通过 `runtime.store`，Node-style 通过 `runtime.store`，Wrap-style 通过 `request.runtime.store` 访问。
- 上一章裁剪历史后可能失去姓名；如果事先把姓名写入 Store，后续会话可通过工具重新检索，而不必一直保留那条聊天。

**来源与核对**：主体来自课件第 3 章；接口、内存写入行为、PostgreSQL 更新语句及运行时注入，按项目当前 `langchain==1.2.12`、`langchain-core==1.2.18`、`langgraph==1.1.2`、`langgraph-checkpoint==4.0.1`、`langgraph-checkpoint-postgres==3.0.5` 的接口和源码核对。没有升级依赖。

官方延伸阅读：`https://docs.langchain.com/oss/python/langchain/long-term-memory`；`https://docs.langchain.com/oss/python/langgraph/persistence`。
